In [2]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import implicit
import scipy.sparse as sparse
from sklearn.metrics import ndcg_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")
print(f"Implicit version: {implicit.__version__}")

c:\Users\Gurveer\anaconda3\envs\ds-portfolio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful
Implicit version: 0.7.2


In [3]:
# Cell 2 — Generate Implicit Feedback Dataset
# Simulates Netflix/Amazon style user-item interactions
np.random.seed(42)

n_users  = 1000
n_items  = 500
n_genres = 8

genres = ['Action', 'Comedy', 'Drama', 'Sci-Fi',
          'Thriller', 'Romance', 'Documentary', 'Animation']

# Item metadata
items_df = pd.DataFrame({
    'item_id':    range(n_items),
    'title':      [f'Movie_{i:04d}' for i in range(n_items)],
    'genre':      np.random.choice(genres, n_items),
    'popularity': np.random.exponential(50, n_items).astype(int) + 1
})

# User preferences — each user has 2 preferred genres
user_prefs = {u: np.random.choice(genres, 2, replace=False) 
              for u in range(n_users)}

# Generate interactions with implicit feedback
interactions = []
for user_id in range(n_users):
    preferred = user_prefs[user_id]
    
    # Interactions with preferred genres (higher confidence)
    pref_items = items_df[items_df['genre'].isin(preferred)]['item_id'].values
    n_pref = np.random.randint(10, 40)
    for item_id in np.random.choice(pref_items, min(n_pref, len(pref_items)), replace=False):
        interactions.append({
            'user_id':   user_id,
            'item_id':   item_id,
            'plays':     np.random.randint(3, 20),  # high engagement
            'rating':    np.random.uniform(3.5, 5.0)
        })
    
    # Random exploration (lower confidence)
    other_items = items_df[~items_df['genre'].isin(preferred)]['item_id'].values
    n_rand = np.random.randint(2, 10)
    for item_id in np.random.choice(other_items, min(n_rand, len(other_items)), replace=False):
        interactions.append({
            'user_id':   user_id,
            'item_id':   item_id,
            'plays':     np.random.randint(1, 4),   # low engagement
            'rating':    np.random.uniform(1.5, 3.5)
        })

interactions_df = pd.DataFrame(interactions).drop_duplicates(
    subset=['user_id', 'item_id']
)

print(f"Users:          {n_users:,}")
print(f"Items:          {n_items:,}")
print(f"Interactions:   {len(interactions_df):,}")
print(f"Sparsity:       {1 - len(interactions_df)/(n_users*n_items):.3%}")
print(f"\nAvg plays per interaction: {interactions_df['plays'].mean():.2f}")
print(f"Avg rating:                {interactions_df['rating'].mean():.2f}")
print(f"\nGenre distribution:")
print(items_df['genre'].value_counts().to_string())

Users:          1,000
Items:          500
Interactions:   29,750
Sparsity:       94.050%

Avg plays per interaction: 9.34
Avg rating:                3.93

Genre distribution:
genre
Sci-Fi         76
Animation      72
Action         71
Drama          64
Romance        62
Documentary    58
Thriller       51
Comedy         46


In [4]:
# Cell 3 — Build Sparse Matrix & Train/Test Split
# Create user-item confidence matrix
# Confidence = 1 + alpha * plays (standard ALS formulation)
alpha = 40

# Full sparse matrix
rows = interactions_df['user_id'].values
cols = interactions_df['item_id'].values
data = 1 + alpha * interactions_df['plays'].values

user_item_matrix = sparse.csr_matrix(
    (data, (rows, cols)),
    shape=(n_users, n_items)
)

# Train/test split — hold out 20% of interactions per user
train_data = interactions_df.sample(frac=0.8, random_state=42)
test_data  = interactions_df.drop(train_data.index)

train_rows = train_data['user_id'].values
train_cols = train_data['item_id'].values
train_vals = 1 + alpha * train_data['plays'].values

train_matrix = sparse.csr_matrix(
    (train_vals, (train_rows, train_cols)),
    shape=(n_users, n_items)
)

print(f"Full matrix shape:   {user_item_matrix.shape}")
print(f"Non-zero entries:    {user_item_matrix.nnz:,}")
print(f"Training entries:    {train_matrix.nnz:,}")
print(f"Test interactions:   {len(test_data):,}")
print(f"\nMatrix density: {user_item_matrix.nnz / (n_users * n_items):.3%}")

Full matrix shape:   (1000, 500)
Non-zero entries:    29,750
Training entries:    23,800
Test interactions:   5,950

Matrix density: 5.950%


In [5]:
# Cell 4 — Train ALS Model (Alternating Least Squares)
# ALS is the industry standard for implicit feedback recommenders
# Used by Netflix, Spotify, and Amazon internally

model_als = implicit.als.AlternatingLeastSquares(
    factors=64,          # latent dimensions
    regularization=0.1,
    iterations=30,
    calculate_training_loss=True,
    random_state=42
)

print("Training ALS model...")
model_als.fit(train_matrix)
print("Training complete")

# Inspect learned factors
user_factors = model_als.user_factors
item_factors = model_als.item_factors

print(f"\nUser factor matrix: {user_factors.shape}")
print(f"Item factor matrix: {item_factors.shape}")
print(f"Latent dimensions:  {user_factors.shape[1]}")
print(f"\nSample user vector (User 0):")
print(user_factors[0].round(4))

Training ALS model...


100%|██████████| 30/30 [00:00<00:00, 82.57it/s, loss=0.00999]

Training complete

User factor matrix: (1000, 64)
Item factor matrix: (500, 64)
Latent dimensions:  64

Sample user vector (User 0):
[ 0.5108  2.7941  5.0005  1.0228 -1.6468 -2.6485  2.9493 -1.9788  2.0181
 -1.0588 -0.5373  1.8213  0.5056  2.2836  0.6221  5.6877  0.9202  0.5321
 -2.9959  4.3377  4.7538  1.7827  4.3531  0.4366 -0.5713  2.8957  1.7623
 -0.1665  0.0809  3.0943 -0.5552 -0.6706  1.8866  3.9217  3.5554 -1.3937
  4.9824 -2.1481 -2.0043  1.9178  0.1666  3.5735  2.8141  0.7742 -1.7566
 -2.5561 -3.7211 -0.1208  1.6116  3.7999 -0.367   3.0413  0.903  -2.4177
  4.3681 -4.2059  3.2302  2.8115 -0.5857  2.4093  1.6886  0.512   2.5138
 -0.4365]


In [6]:
# Cell 5 — Generate Recommendations & Evaluate
def get_recommendations(user_id, n=10):
    """Get top-N recommendations for a user"""
    ids, scores = model_als.recommend(
        user_id, train_matrix[user_id],
        N=n, filter_already_liked_items=True
    )
    recs = items_df.iloc[ids].copy()
    recs['score'] = scores
    recs['preferred_genres'] = str(list(user_prefs[user_id]))
    return recs[['item_id', 'title', 'genre', 'score', 'preferred_genres']]

# Show recommendations for 3 sample users
for user_id in [0, 42, 99]:
    print(f"\n=== Recommendations for User {user_id} ===")
    print(f"Preferred genres: {list(user_prefs[user_id])}")
    recs = get_recommendations(user_id, n=5)
    print(recs.to_string(index=False))

# Evaluate NDCG@10 on test set
print("\n\n=== Evaluating NDCG@10 ===")
ndcg_scores = []
test_users  = test_data['user_id'].unique()[:200]

for user_id in test_users:
    test_items = test_data[test_data['user_id']==user_id]['item_id'].values
    if len(test_items) == 0:
        continue

    ids, scores = model_als.recommend(
        user_id, train_matrix[user_id],
        N=10, filter_already_liked_items=True
    )

    # Binary relevance: 1 if recommended item is in test set
    relevance = np.array([1 if i in test_items else 0 for i in ids])

    if relevance.sum() > 0:
        ideal = np.sort(relevance)[::-1]
        ndcg  = ndcg_score([ideal], [relevance])
        ndcg_scores.append(ndcg)

print(f"Users evaluated:  {len(ndcg_scores)}")
print(f"NDCG@10:          {np.mean(ndcg_scores):.4f}")
print(f"Min NDCG:         {np.min(ndcg_scores):.4f}")
print(f"Max NDCG:         {np.max(ndcg_scores):.4f}")


=== Recommendations for User 0 ===
Preferred genres: [np.str_('Action'), np.str_('Comedy')]
 item_id      title     genre    score                       preferred_genres
     362 Movie_0362    Action 1.347987 [np.str_('Action'), np.str_('Comedy')]
     292 Movie_0292    Action 1.001099 [np.str_('Action'), np.str_('Comedy')]
     215 Movie_0215  Thriller 0.889184 [np.str_('Action'), np.str_('Comedy')]
     203 Movie_0203    Action 0.865170 [np.str_('Action'), np.str_('Comedy')]
     271 Movie_0271 Animation 0.789198 [np.str_('Action'), np.str_('Comedy')]

=== Recommendations for User 42 ===
Preferred genres: [np.str_('Documentary'), np.str_('Action')]
 item_id      title       genre    score                            preferred_genres
     148 Movie_0148      Action 0.906754 [np.str_('Documentary'), np.str_('Action')]
     397 Movie_0397     Romance 0.779508 [np.str_('Documentary'), np.str_('Action')]
      58 Movie_0058       Drama 0.718844 [np.str_('Documentary'), np.str_('Action')]


In [7]:
# Cell 6 — Visualize & Export
# Genre hit rate — how often recommendations match user preferences
genre_matches = []
for user_id in range(200):
    preferred = set(user_prefs[user_id])
    recs = get_recommendations(user_id, n=10)
    match_rate = recs['genre'].isin(preferred).mean()
    genre_matches.append({
        'user_id':    user_id,
        'match_rate': match_rate,
        'preferred':  str(list(preferred))
    })

match_df = pd.DataFrame(genre_matches)
print(f"Avg genre match rate: {match_df['match_rate'].mean():.3%}")

# Plot score distribution
fig1 = px.histogram(match_df, x='match_rate', nbins=20,
                    title='Genre Preference Match Rate — ALS Recommendations',
                    labels={'match_rate': 'Fraction of top-10 matching preferred genres'},
                    template='plotly_dark')
fig1.update_layout(width=750, height=400)
fig1.show()

# Plot item popularity vs recommendation frequency
rec_counts = []
for user_id in range(500):
    ids, _ = model_als.recommend(
        user_id, train_matrix[user_id],
        N=10, filter_already_liked_items=True
    )
    rec_counts.extend(ids)

rec_freq = pd.Series(rec_counts).value_counts().reset_index()
rec_freq.columns = ['item_id', 'rec_count']
rec_freq = rec_freq.merge(items_df[['item_id','genre','popularity']], on='item_id')

fig2 = px.scatter(rec_freq, x='popularity', y='rec_count',
                  color='genre', size='rec_count',
                  title='Item Popularity vs Recommendation Frequency by Genre',
                  template='plotly_dark')
fig2.update_layout(width=850, height=500)
fig2.show()

# Export
import os
output_dir = r'C:\Users\Gurveer\ds-portfolio\project-08-recommender\outputs'
os.makedirs(output_dir, exist_ok=True)
match_df.to_csv(f'{output_dir}\\genre_match_rates.csv', index=False)
rec_freq.to_csv(f'{output_dir}\\recommendation_frequency.csv', index=False)
items_df.to_csv(f'{output_dir}\\items_metadata.csv', index=False)
pd.DataFrame({'metric': ['NDCG@10', 'Genre Match Rate'],
              'value': [round(np.mean(ndcg_scores), 4),
                        round(match_df['match_rate'].mean(), 4)]
              }).to_csv(f'{output_dir}\\eval_metrics.csv', index=False)
print(f"\nExports saved to outputs/")

Avg genre match rate: 78.350%



Exports saved to outputs/
